# MedWear Paper Reproduction (One-Click)

**Pipeline:** synthetic data (`seed=42`) → BHI → MAD anomaly → ONNX inference → XAI charts

Requires Node.js + `npm ci` at repo root. Run: `pip install -r notebooks/requirements.txt`

> BHI = behavioral health index (not disease risk). MAD = robust heuristic (not clinical validation).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Repo root (parent of notebooks/)
ROOT = Path.cwd()
if not (ROOT / "server.js").exists():
    ROOT = ROOT.parent
assert (ROOT / "server.js").exists(), "Run from repo root or notebooks/ directory"

OUT = ROOT / "notebooks" / "output"
OUT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Repo:", ROOT)
print("Output:", OUT)

def run_node(args, *, input_text=None, quiet=False):
    kw = dict(cwd=ROOT, text=True, capture_output=True)
    if input_text is not None:
        kw["input"] = input_text
    r = subprocess.run(["node"] + args, **kw)
    if r.returncode != 0:
        print(r.stderr, file=sys.stderr)
        raise RuntimeError(f"node {' '.join(args)} failed")
    if not quiet and r.stdout.strip():
        print(r.stdout.strip())
    return r.stdout

def bridge_cases(cases):
    out = run_node(["scripts/paper_reproduction_bridge.js"], input_text=json.dumps({"cases": cases}), quiet=True)
    return json.loads(out)


## 1. Generate / load synthetic data (seed=42)

In [ ]:
SEED = 42
N_CASES = 5000
DATASET = ROOT / "benchmarks" / "wearable-analytics-dataset.json"

if not DATASET.exists():
    print("Generating synthetic benchmark (seed=42)...")
    run_node(["scripts/generate-wearable-benchmark.js", "--n", str(N_CASES), "--seed", str(SEED)])
else:
    print(f"Using existing dataset: {DATASET}")

raw = json.loads(DATASET.read_text(encoding="utf-8"))
cases = raw["cases"]
print(f"Loaded {len(cases)} cases · version {raw.get('version', '?')}")

# Representative demo cases for visualization
demo_ids = ["WA-001", "WA-010", "WA-050", "WA-100"]
demo_cases = [c for c in cases if c["id"] in demo_ids] or cases[:4]


## 2. BHI index + MAD anomaly detection (product engine via Node bridge)

In [ ]:
print("Running MedWear engine bridge (BHI + MAD + features)...")
bridged = bridge_cases(cases[:500])  # aggregate stats on 500 cases
rows = []
for item in bridged["cases"]:
    rows.append({
        "id": item["id"],
        "bhi": item["bhi"],
        "bhiTier": item["bhiTier"],
        "madCount": item["madAnomalyCount"],
        "trendDelta": item.get("trendDelta", 0),
        "goldTier": (item.get("expected") or {}).get("riskLevel"),
    })
summary_df = pd.DataFrame(rows)
print(summary_df["bhiTier"].value_counts())
print("MAD flagged:", (summary_df["madCount"] > 0).mean().round(3), "of cases")

demo = bridge_cases(demo_cases)["cases"]
focus = demo[0]
print(f"\nFocus case: {focus['id']} · BHI={focus['bhi']} · tier={focus['bhiTier']} · MAD anomalies={focus['madAnomalyCount']}")


## 3. XAI — BHI component visualization

In [ ]:
# --- XAI: BHI component decomposition ---
comp = focus["bhiComponents"]
weights = focus["bhiWeights"]
labels = list(comp.keys())
values = [comp[k] for k in labels]
w = [weights.get(k, 0) for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(labels, values, color=sns.color_palette("Blues_d", len(labels)))
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Component score (0–1)")
axes[0].set_title(f"BHI components · {focus['id']} (score={focus['bhi']})")

axes[1].pie(w, labels=[f"{k}\n({weights.get(k, 0):.0%})" for k in labels], autopct="%1.0f%%", startangle=90)
axes[1].set_title("Fusion weights (transparent rule engine)")

fig.suptitle("XAI — Behavioral Health Index decomposition", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "xai_bhi_components.png", bbox_inches="tight")
plt.show()

# Tier distribution
fig, ax = plt.subplots(figsize=(6, 4))
order = ["low", "moderate", "high", "unknown"]
summary_df["bhiTier"].value_counts().reindex(order).fillna(0).plot(kind="bar", ax=ax, color="#2E7D32")
ax.set_title("BHI watch-tier distribution (n=500 sample)")
ax.set_xlabel("Tier")
ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(OUT / "bhi_tier_distribution.png", bbox_inches="tight")
plt.show()


## 4. MAD robust anomaly — baseline vs readings

In [ ]:
# --- MAD baseline vs target-day readings ---
case_raw = next(c for c in demo_cases if c["id"] == focus["id"])
target_day = focus["targetDay"]
day = case_raw["days"][target_day]
hrs = day.get("heartRate", [])
spo2 = day.get("spo2", [])
base = focus.get("madBaseline") or {}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if hrs:
    axes[0].axhline(base.get("hrMedian", np.median(hrs)), color="#1565C0", ls="--", label="HR baseline median")
    thr = base.get("hrMedian", np.median(hrs)) + 2.5 * (base.get("hrMadScaled", 5) / 1.4826)
    axes[0].axhline(thr, color="#C62828", ls=":", label="MAD spike threshold")
    axes[0].plot(range(len(hrs)), hrs, "o-", color="#EF6C00")
    axes[0].set_title(f"HR readings · {target_day}")
    axes[0].set_xlabel("Reading index")
    axes[0].legend(fontsize=8)
if spo2:
    axes[1].axhline(base.get("spo2Median", np.median(spo2)), color="#1565C0", ls="--", label="SpO₂ median")
    axes[1].plot(range(len(spo2)), spo2, "s-", color="#00838F")
    axes[1].set_title(f"SpO₂ readings · {target_day}")
    axes[1].set_xlabel("Reading index")
    axes[1].legend(fontsize=8)

anom_text = "; ".join(a.get("type", a.get("rule", "?")) for a in focus.get("madAnomalies", [])) or "none"
fig.suptitle(f"MAD heuristic · anomalies: {anom_text}", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "mad_baseline_readings.png", bbox_inches="tight")
plt.show()


## 5. ONNX local inference (medwear_rf.onnx)

In [ ]:
import onnxruntime as ort

ONNX_PATH = ROOT / "server" / "ai" / "models" / "medwear_rf.onnx"
META_PATH = ROOT / "server" / "ai" / "models" / "medwear_rf.meta.json"
FEATURES_CSV = ROOT / "experiments" / "data" / "medwear" / "features_v1.csv"

if not FEATURES_CSV.exists():
    print("Exporting features...")
    run_node(["scripts/export_features.js", "--input", str(DATASET.relative_to(ROOT)), "--out", str(FEATURES_CSV.relative_to(ROOT))])

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
cols = meta["feature_cols"]
classes = meta["label_classes"]

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
inp_name = sess.get_inputs()[0].name
out_name = next(o.name for o in sess.get_outputs() if "label" in o.name)

# Batch inference on demo cases
vecs = np.array([d["featureVector"] for d in demo], dtype=np.float32)
t0 = __import__("time").perf_counter()
pred_idx = sess.run([out_name], {inp_name: vecs})[0]
onnx_ms = (__import__("time").perf_counter() - t0) * 1000
onnx_labels = [classes[int(i)] for i in pred_idx]

onnx_df = pd.DataFrame({
    "id": [d["id"] for d in demo],
    "bhiTier_rule": [d["bhiTier"] for d in demo],
    "onnx_tier": onnx_labels,
    "bhi": [d["bhi"] for d in demo],
})
print(onnx_df)
print(f"ONNX batch-{len(demo)} latency: {onnx_ms:.2f} ms")

# Latency curve by batch size
batch_sizes = [1, 4, 8, 16, 32, 64]
pool = np.array([c["featureVector"] for c in bridge_cases(cases[:256])["cases"]], dtype=np.float32)
latencies = []
for b in batch_sizes:
    x = pool[:b]
    _ = sess.run([out_name], {inp_name: x})  # warmup
    ts = []
    for _ in range(20):
        t0 = __import__("time").perf_counter()
        sess.run([out_name], {inp_name: x})
        ts.append((__import__("time").perf_counter() - t0) * 1000)
    latencies.append(np.mean(ts))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(batch_sizes, latencies, "o-", color="#1565C0", linewidth=2)
ax.set_xscale("log", base=2)
ax.set_xlabel("Batch size")
ax.set_ylabel("Mean latency (ms)")
ax.set_title("ONNX Runtime local inference (medwear_rf.onnx)")
fig.tight_layout()
fig.savefig(OUT / "onnx_latency_by_batch.png", bbox_inches="tight")
plt.show()


## 6. XAI — SHAP feature attribution (fair 15-dim RF)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import shap
except ImportError:
    raise ImportError("pip install shap>=0.43")

FAIR_COLS = [c for c in cols if c not in ("anomaly_flag", "health_score_norm")]
sample_n = 400
sample_cases = cases[:sample_n]
feat_rows = bridge_cases(sample_cases)["cases"]
X = pd.DataFrame([dict(zip(FAIR_COLS, [r["featureVector"][cols.index(c)] for c in FAIR_COLS])) for r in feat_rows])
y = pd.Series([r["bhiTier"] for r in feat_rows])

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=120, random_state=SEED, class_weight="balanced", n_jobs=-1)),
])
pipe.fit(X, y)

X_scaled = pipe.named_steps["scaler"].transform(pipe.named_steps["imputer"].transform(X))
clf = pipe.named_steps["clf"]
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_scaled)

# Summary bar (mean |SHAP| across samples & classes)
if isinstance(shap_values, list):
    sv = np.mean([np.abs(s).mean(axis=0) for s in shap_values], axis=0)
elif getattr(shap_values, "ndim", 0) == 3:
    sv = np.abs(shap_values).mean(axis=(0, 2))
else:
    sv = np.abs(shap_values).mean(axis=0)

order = np.argsort(sv)[::-1][:12]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(np.array(FAIR_COLS)[order][::-1], sv[order][::-1], color="#7B1FA2")
ax.set_xlabel("mean |SHAP value|")
ax.set_title(f"XAI — RF feature attribution (fair {len(FAIR_COLS)}-dim, n={sample_n})")
fig.tight_layout()
fig.savefig(OUT / "xai_shap_feature_importance.png", bbox_inches="tight")
plt.show()

# Beeswarm (first class layer; handles list or 3D ndarray from TreeExplainer)
if isinstance(shap_values, list):
    shap_layer = shap_values[0]
elif getattr(shap_values, "ndim", 0) == 3:
    shap_layer = shap_values[:, :, 0]
else:
    shap_layer = shap_values
shap.summary_plot(shap_layer, pd.DataFrame(X_scaled, columns=FAIR_COLS), show=False, max_display=12)
plt.gcf().savefig(OUT / "xai_shap_beeswarm.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved XAI charts to", OUT)


## Summary

| Step | Engine | Output |
|------|--------|--------|
| Synthetic data | `generate-wearable-benchmark.js` seed=42 | `benchmarks/wearable-analytics-dataset.json` |
| BHI | `behavioralHealthIndex.js` | Component scores + watch tier |
| MAD | `robustAnomaly.js` | Baseline + spike rules |
| ONNX | `medwear_rf.onnx` | Tier label + latency curve |
| XAI | SHAP + BHI bars | `notebooks/output/*.png` |

Full evaluation: `npm run evaluate` · System benchmarks: `npm run benchmark:system`